Data Processing for Tethered Balloon

In [1]:
# imports
import math
import numpy as np
import plotly as plt
import pandas as pd
from datetime import datetime

In [2]:
# Constants + Strain Gauge Calibration
offset_0 = 406.62
offset_1 = -466.48
offset_2 = 246.45
offset_3 = 313.22

scale_0 = 0.001
scale_1 = 0.001
scale_2 = 0.001
scale_3 = 0.001

In [3]:
def calibrate_tether_data(filename):
    # Load CSV data from four_tether folders and calibrate.
    
    df = pd.read_csv(filename, header=None, names=['time', 'raw0', 'raw1', 'raw2', 'raw3'])
    
    df['T0'] = ((df['raw0'] * scale_0) + offset_0) * math.sin(math.radians(55))
    df['T1'] = ((df['raw1'] * scale_1) + offset_1) * math.sin(math.radians(55))
    df['T2'] = ((df['raw2'] * scale_2) + offset_2) * math.sin(math.radians(55))
    df['T3'] = ((df['raw3'] * scale_3) + offset_3) * math.sin(math.radians(55))
    
    df['time'] = (df['time'] - df['time'][0])/1000

    df = df[['time', 'T0', 'T1', 'T2', 'T3']]
    
    return df

In [4]:

df_no_net_5 = calibrate_tether_data("../four_tether_no_net_7_1/four_tether_5%_no_net.csv")
print(df_no_net_5.head())


    time         T0        T1        T2         T3
0  0.000  17.737099 -6.137906  6.388567  99.424579
1  0.108  17.513471 -6.133811  6.612195  99.415569
2  0.216  17.340630 -6.116608  6.863675  99.513048
3  0.324  17.094065 -6.146917  7.029963  99.544995
4  0.432  16.763947 -6.049438  7.176591  99.550729


In [5]:
def plot_tether_data(df, title):
    # Plot tether tension data over time
    
    import plotly.graph_objects as go
    
    fig = go.Figure()
    
    tension_columns = [col for col in df.columns if col != 'time']
    
    for col in tension_columns:
        fig.add_trace(go.Scatter(
            x=df['time'],
            y=df[col],
            mode='lines',
            name=col,
            line=dict(width=2)
        ))
    
    fig.update_layout(
        title=title,
        xaxis_title='Time (s)',
        yaxis_title='Tension (g)',
        template='plotly_white',
        hovermode='x unified',
        height=500,
        width=1000
    )
    
    fig.show()
    return fig


In [6]:
# Plot the data
plot_tether_data(df_no_net_5, title="5% Duty Cycle Without Net")

In [7]:
from scipy import signal
from scipy.fft import fft, fftfreq

def calculate_frequency_over_time(df, column, window_size=0.5, min_height=None):
    """
    Calculate instantaneous frequency over time using peak detection.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        DataFrame with 'time' and data columns
    column : str
        Column name to analyze (e.g., 'T0')
    window_size : float
        Time window (seconds) to use for frequency calculation
    min_height : float
        Minimum height for peak detection (default: auto)
    
    Returns:
    --------
    dict with 'time_windows', 'frequencies', 'peak_times'
    """
    # Calculate time increment
    dt = df['time'].diff().mean()
    
    # Find peaks in the signal
    if min_height is None:
        min_height = df[column].std()
    
    peaks, properties = signal.find_peaks(df[column], height=min_height, distance=1/dt)
    
    time_windows = []
    frequencies = []
    
    # Calculate frequency in sliding windows
    window_samples = int(window_size / dt)
    
    for i in range(len(df) - window_samples):
        window_peaks = peaks[(peaks >= i) & (peaks < i + window_samples)]
        
        if len(window_peaks) > 1:
            # Calculate frequency from peak spacing
            time_diff = (peaks[window_peaks[-1]] - peaks[window_peaks[0]]) * dt
            num_cycles = len(window_peaks) - 1
            
            if time_diff > 0:
                freq = num_cycles / time_diff
                time_windows.append(df['time'].iloc[i + window_samples // 2])
                frequencies.append(freq)
    
    return {
        'time_windows': time_windows,
        'frequencies': frequencies,
        'peak_times': df['time'].iloc[peaks].values,
        'peak_values': df[column].iloc[peaks].values
    }

def plot_frequency_over_time(df, column, window_size=0.5, title=None):
    """Plot frequency over time to check regularity."""
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    
    if title is None:
        title = f"Frequency Analysis for {column}"
    
    freq_data = calculate_frequency_over_time(df, column, window_size)
    
    # Create subplots: tension data on top, frequency on bottom
    fig = make_subplots(
        rows=2, cols=1,
        subplot_titles=("Raw Signal with Peaks", "Frequency Over Time"),
        row_heights=[0.6, 0.4]
    )
    
    # Plot raw signal
    fig.add_trace(
        go.Scatter(x=df['time'], y=df[column], name='Signal', line=dict(color='blue')),
        row=1, col=1
    )
    
    # Plot peaks
    fig.add_trace(
        go.Scatter(x=freq_data['peak_times'], y=freq_data['peak_values'], 
                   mode='markers', name='Peaks', marker=dict(color='red', size=6)),
        row=1, col=1
    )
    
    # Plot frequency
    fig.add_trace(
        go.Scatter(x=freq_data['time_windows'], y=freq_data['frequencies'], 
                   name='Frequency', line=dict(color='green', width=2)),
        row=2, col=1
    )
    
    fig.update_xaxes(title_text="Time (s)", row=2, col=1)
    fig.update_yaxes(title_text="Tension (g)", row=1, col=1)
    fig.update_yaxes(title_text="Frequency (Hz)", row=2, col=1)
    
    fig.update_layout(title_text=title, height=700, hovermode='x unified')
    fig.show()
    
    return freq_data

def compare_frequencies_across_duty_cycles(dataframes_dict, column='T0', window_size=0.5):
    """
    Compare average frequencies across different duty cycles.
    
    Parameters:
    -----------
    dataframes_dict : dict
        Dictionary with duty cycle names as keys and dataframes as values
        e.g., {'5%': df_5, '10%': df_10, ...}
    column : str
        Column to analyze
    window_size : float
        Time window for frequency calculation
    
    Returns:
    --------
    dict with comparison data
    """
    comparison = {}
    
    for duty_cycle, df in dataframes_dict.items():
        freq_data = calculate_frequency_over_time(df, column, window_size)
        avg_freq = np.mean(freq_data['frequencies']) if freq_data['frequencies'] else 0
        std_freq = np.std(freq_data['frequencies']) if freq_data['frequencies'] else 0
        
        comparison[duty_cycle] = {
            'avg_frequency': avg_freq,
            'std_frequency': std_freq,
            'frequencies': freq_data['frequencies'],
            'time_windows': freq_data['time_windows']
        }
    
    return comparison

def plot_frequency_comparison(comparison_dict, title="Frequency Comparison Across Duty Cycles"):
    """Plot box plot and mean frequencies across duty cycles."""
    import plotly.graph_objects as go
    
    fig = go.Figure()
    
    duty_cycles = list(comparison_dict.keys())
    avg_frequencies = [comparison_dict[dc]['avg_frequency'] for dc in duty_cycles]
    std_frequencies = [comparison_dict[dc]['std_frequency'] for dc in duty_cycles]
    
    # Add mean with error bars
    fig.add_trace(go.Bar(
        x=duty_cycles,
        y=avg_frequencies,
        error_y=dict(type='data', array=std_frequencies),
        name='Mean ± Std Dev',
        marker=dict(color='lightblue', line=dict(color='darkblue', width=2))
    ))
    
    fig.update_layout(
        title=title,
        xaxis_title='Duty Cycle',
        yaxis_title='Frequency (Hz)',
        template='plotly_white',
        height=500,
        width=800
    )
    
    fig.show()
    
    # Print summary
    print("\nFrequency Summary:")
    print("-" * 50)
    for dc in duty_cycles:
        print(f"{dc:>8}: {comparison_dict[dc]['avg_frequency']:.2f} ± {comparison_dict[dc]['std_frequency']:.2f} Hz")


In [8]:
# Example: Plot frequency over time for a single dataset
freq_data_5 = plot_frequency_over_time(df_no_net_5, 'T0', window_size=1.0, 
                                       title="T0 Frequency Analysis - 5% Duty Cycle")

In [9]:
# Example: Compare frequencies across multiple duty cycles
# Load data for different duty cycles
df_5 = calibrate_tether_data("../four_tether_no_net_7_1/four_tether_5%_no_net.csv")
df_10 = calibrate_tether_data("../four_tether_no_net_7_1/four_tether_10%_no_net.csv")
df_20 = calibrate_tether_data("../four_tether_no_net_7_1/four_tether_20%_no_net.csv")
df_40 = calibrate_tether_data("../four_tether_no_net_7_1/four_tether_40%_no_net.csv")

# Create comparison dictionary
duty_cycle_data = {
    '5%': df_5,
    '10%': df_10,
    '20%': df_20,
    '40%': df_40
}

# Compare frequencies for T0 across duty cycles
comparison = compare_frequencies_across_duty_cycles(duty_cycle_data, column='T0', window_size=1.0)

# Plot comparison
plot_frequency_comparison(comparison, title="T0 Frequency Comparison Across Duty Cycles (No Net)")


Frequency Summary:
--------------------------------------------------
      5%: 0.00 ± 0.00 Hz
     10%: 0.00 ± 0.00 Hz
     20%: 0.00 ± 0.00 Hz
     40%: 0.00 ± 0.00 Hz
